In [1]:
import torch
import torch.nn as nn 
import torch.nn.functional as F 
import torch.optim as optim 
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms, datasets
import warnings
warnings.filterwarnings("ignore")


In [2]:
#setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

Using: cuda


In [3]:
#data augmentation using transformers.Compose

train_transforms_pipeline = transforms.Compose([
    transforms.ColorJitter(contrast=0.2, brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_transforms_pipeline = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

val_transforms_pipeline = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.MNIST(
    root='./data',
    download=True,
    train=True,
    transform=train_transforms_pipeline
)

test_data = datasets.MNIST(
    root="./data",
    download=True,
    train=False,
    transform=test_transforms_pipeline
)

val_data = datasets.MNIST(
    root="./data",
    download=True,
    train=False,
    transform=val_transforms_pipeline
)

train_loader = DataLoader(train_data, shuffle=True, batch_size=128, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_data, shuffle=False, batch_size=128, num_workers=4, pin_memory=True )
val_loader = DataLoader(val_data, shuffle=False, batch_size=128, num_workers=4, pin_memory=True)



In [4]:
#check the output_dim value
print(f"output_dim value: {train_data.classes}")

output_dim value: ['0 - zero', '1 - one', '2 - two', '3 - three', '4 - four', '5 - five', '6 - six', '7 - seven', '8 - eight', '9 - nine']


In [5]:
# for inputs, labels in train_loader:
#     print(f"Train -> inputs:{inputs.shape} | lables: {labels.shape}")
#     break
# for inputs, labels in test_loader:
#     print(f"Test -> inputs:{inputs.shape} | lables: {labels.shape}")
#     break
# for inputs, labels in val_loader:
#     print(f"Val -> inputs:{inputs.shape} | lables: {labels.shape}")
#     break

data_loaders = [("Train", train_loader), ("Val", val_loader), ("Test", test_loader)]

for name, dataloader in data_loaders:
    for inputs, labels in dataloader:
        print(f"[{name}] inputs:{inputs.shape} | labels:{labels.shape}")
        break

[Train] inputs:torch.Size([128, 1, 28, 28]) | labels:torch.Size([128])
[Val] inputs:torch.Size([128, 1, 28, 28]) | labels:torch.Size([128])
[Test] inputs:torch.Size([128, 1, 28, 28]) | labels:torch.Size([128])


In [6]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classification = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*3*3, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 10)  # MNIST has 10 classes
        )

    def forward(self, x):
        #return self.classification(self.features(x))
        x = self.features(x)
        x = self.classification(x)
        return x

model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (features): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout2d(p=0.25, inplace=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (15): Conv2d(128

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=50)

In [8]:

#TRAINING LOOP
num_epochs = 10

for epoch in range(num_epochs):

    # ======== TRAINING ========
    model.train()
    train_loss, train_correct = 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()

    avg_train_loss = train_loss / len(train_loader.dataset)
    avg_train_acc  = train_correct / len(train_loader.dataset)

    # ======== VALIDATION ========
    model.eval()
    val_loss, val_correct = 0, 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()

    avg_val_loss = val_loss / len(val_loader.dataset)   
    avg_val_acc  = val_correct / len(val_loader.dataset)

    scheduler.step()  

    # ======== PRINT RESULTS ========
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train -> Loss: {avg_train_loss:.4f} | Acc: {avg_train_acc:.4f}")
    print(f"  Val   -> Loss: {avg_val_loss:.4f} | Acc: {avg_val_acc:.4f}")
    print("-" * 40)



#TEST (once, after training)
# =======================
model.eval()
test_loss, test_correct = 0, 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * inputs.size(0)
        test_correct += (outputs.argmax(1) == labels).sum().item()

avg_test_loss = test_loss / len(test_loader.dataset)
avg_test_acc  = test_correct / len(test_loader.dataset)

print("=" * 40)
print("🧪 Final Test Results")
print(f"  Test Loss: {avg_test_loss:.4f} | Test Acc: {avg_test_acc:.4f}")
print("=" * 40)

Epoch 1/10
  Train -> Loss: 0.2480 | Acc: 0.9392
  Val   -> Loss: 0.0424 | Acc: 0.9882
----------------------------------------
Epoch 2/10
  Train -> Loss: 0.0528 | Acc: 0.9873
  Val   -> Loss: 0.0231 | Acc: 0.9928
----------------------------------------
Epoch 3/10
  Train -> Loss: 0.0335 | Acc: 0.9915
  Val   -> Loss: 0.0202 | Acc: 0.9934
----------------------------------------
Epoch 4/10
  Train -> Loss: 0.0252 | Acc: 0.9933
  Val   -> Loss: 0.0184 | Acc: 0.9940
----------------------------------------
Epoch 5/10
  Train -> Loss: 0.0204 | Acc: 0.9945
  Val   -> Loss: 0.0165 | Acc: 0.9936
----------------------------------------
Epoch 6/10
  Train -> Loss: 0.0154 | Acc: 0.9962
  Val   -> Loss: 0.0151 | Acc: 0.9950
----------------------------------------
Epoch 7/10
  Train -> Loss: 0.0134 | Acc: 0.9966
  Val   -> Loss: 0.0166 | Acc: 0.9946
----------------------------------------
Epoch 8/10
  Train -> Loss: 0.0110 | Acc: 0.9974
  Val   -> Loss: 0.0154 | Acc: 0.9946
-----------------